# Analysis AI-Team

### Load and import 

In [1]:
import os, sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))

# --- General libraries --------------------------------------------------
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px          # interactive world map
import plotly.graph_objects as go    # confusion arcs, custom figures
from IPython.display import display
import torch

# --- Shared setup -------------------------------------------------------
from shared.data import Config, CountryMapper, OSV5MDataset, MetadataDecoder
from shared.models import (GeoCLIPModel, PredictionRunner, Segmenter, TextDetector,
                           Inpainter, fit_image, fit_mask, free_gpu_memory, get_device)
from shared.masking import MaskApplier
cfg = Config()

# --- Load groups own py files -------------------------------------------------------
# from pair_a.<file> import ...  


print("PyTorch", torch.__version__, "| CUDA build", torch.version.cuda)
device = get_device(cfg.device)
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    torch.cuda.reset_peak_memory_stats()
    
# ---------------- change settings here ----------------
cfg.images_per_country = 50
cfg.use_clip_baseline = False        # True -> downloads the original CLIP once (~1.7 GB)
cfg.crop_mode = "center_crop"        # "center_crop" | "pad" | "squash"
cfg.candidate_list = "v1"            # "v1" | "all_test_locations"
cfg.fill_mode = "gray"               # "gray" | "blur" | "inpaint" (inpaint downloads LaMa, ~200 MB)
cfg.batch_size = 32
cfg.num_workers = 4                  # set to 0 if image loading hangs (e.g. on Windows)
# ------------------------------------------------------

cfg.check()
cfg.show()

mapper = CountryMapper("resources/country_aliases.csv")
dataset = OSV5MDataset(cfg, mapper).load()      # no download, already on disk
decoder = MetadataDecoder()

run_name = f"streetclip_{cfg.candidate_list}_{cfg.crop_mode}"   # streetclip_v1_center_crop
preds, logits, embeds, info = PredictionRunner.load(f"results/shared/{run_name}")
preds = decoder.decode(preds)        # adds climate / land-cover / drive-side names

pilot_ids = dataset.build_pilot()    # loads the shared resources/pilot_ids.csv
print(f"{len(preds):,} predictions | {len(pilot_ids):,} pilot images | model: {info['model']}")

c:\Users\manue\Documents\Uni\Master\Semester 3\AI and Vis\Github\ai_and_visualisation_OpenSI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


PyTorch 2.11.0+cu128 | CUDA build 12.8
GPU: NVIDIA GeForce RTX 5070
seed                   42
device                 auto
data_dir               data
results_dir            results
resources_dir          resources
images_per_country     50
streetclip_model       geolocal/StreetCLIP
use_clip_baseline      False
clip_model             openai/clip-vit-large-patch14-336
prompt_template        A Street View photo in {country}.
candidate_list         v1
crop_mode              center_crop
batch_size             32
num_workers            4
half_precision         True
top_k                  10
segmentation_model     facebook/mask2former-swin-large-mapillary-vistas-semantic
fill_mode              gray
blur_radius            25
mask_dilation_px       5
num_random_controls    3
lama_url               https://github.com/enesmsahin/simple-lama-inpainting/releases/download/v0.1.0/big-lama.pt
210,122 predictions | 9,578 pilot images | model: geolocal/StreetCLIP


## Q2: Overconfidence and training coverage
When is the model confidently wrong, and can we infer which countries were in its Street View training?

In [ ]:
from pair_a import q2_confidence as q2

countries = mapper.table
clip_preds = pd.read_parquet("results/shared/clip_v1_center_crop/predictions.parquet")   # original CLIP

### Q2.1 Calibration
- Is the top-1 probability an honest estimate of the accuracy?
- StreetCLIP vs. original CLIP (same model before the Street View fine-tuning)

In [ ]:
cal = {"StreetCLIP": q2.calibration_table(preds), "CLIP": q2.calibration_table(clip_preds)}
display(cal["StreetCLIP"].round(3))
q2.plot_reliability(cal);

### Q2.2 Where is the model confidently wrong?
- Per country: gap = mean confidence - accuracy (gap > 0: overconfident)
- Confident: top-1 probability >= 0.8

In [ ]:
sc_table = q2.country_table(preds, countries)
display(sc_table[~sc_table.low_n].sort_values("gap", ascending=False).head(10).round(3))
q2.plot_gap_map(sc_table).show()

In [ ]:
display(q2.confident_errors(preds, countries).round(3))

- Low precision: a label the model picks often, but wrongly
- n_true = 0: country has no test images, so every prediction of it is an error

In [ ]:
pred_table = q2.predicted_table(preds, countries)
display(pred_table[pred_table.n_pred >= 100].sort_values("precision").head(10).round(3))

### Q2.3 Training coverage
- Paper's training list: uncertain evidence (says 101 countries, lists 92)
- Idea: countries in the Street View training should gain most from fine-tuning (CLIP -> StreetCLIP)

In [ ]:
display(q2.training_groups(sc_table).round(3))
gain = q2.gain_table(sc_table, q2.country_table(clip_preds, countries))
q2.plot_gain(gain).show()

In [ ]:
big = gain[~gain.low_n]
display(big[~big.in_train_list].sort_values("gain", ascending=False).head(8).round(3))   # big gain, not in list
display(big[big.in_train_list].sort_values("gain").head(8).round(3))                     # in list, but no gain

## Q4: Environment
Do accuracy and confidence vary with climate, land cover, distance to the sea, or urban vs. rural setting?

In [ ]:
from pair_a import q4_environment as q4